# Inspect MAG7 Fundamentals and Information Availability

This notebook downloads the private `cookekieran/mag7-fundamentals` dataset and answers three questions:

1. What fundamentals data is available?
2. What fiscal periods and public report dates does it cover?
3. When can each quarterly statement safely become available to a predictive model?

The key leakage distinction is:

- `fiscal_period_end`: the quarter described by the statement
- `reported_date`: the date the earnings report became public
- `available_from_date`: the first date on which the model is permitted to use it

Joining fundamentals to prices using only `fiscal_period_end` would leak future information.

In [1]:
import json
import os
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from huggingface_hub import HfApi, hf_hub_download
from IPython.display import display

pd.set_option('display.max_columns', 100)
pd.set_option('display.max_colwidth', 160)

load_dotenv(Path('.env'))
HF_TOKEN = os.getenv('HF_TOKEN')
assert HF_TOKEN, 'HF_TOKEN is missing from .env'

REPO_ID = 'cookekieran/mag7-fundamentals'
api = HfApi(token=HF_TOKEN)
repo_info = api.repo_info(REPO_ID, repo_type='dataset')

print('Repository:', REPO_ID)
print('Private:', repo_info.private)
print('Revision inspected:', repo_info.sha)

Repository: cookekieran/mag7-fundamentals
Private: True
Revision inspected: de97de7a8efc130a258fb295a06520f9a8fed30f


## Download canonical tables

The notebook pins every download to the displayed repository revision, making this inspection reproducible even if the repository is updated later.

In [2]:
REVISION = repo_info.sha

def download(repo_path):
    return hf_hub_download(
        repo_id=REPO_ID,
        repo_type='dataset',
        filename=repo_path,
        revision=REVISION,
        token=HF_TOKEN,
    )

statements_tidy = pd.read_parquet(download('data/statements_quarterly_tidy.parquet'))
statements_wide = pd.read_parquet(download('data/statements_quarterly_wide.parquet'))
earnings = pd.read_parquet(download('data/earnings_dates_quarterly.parquet'))
model_ready = pd.read_parquet(download('data/model_ready_fundamentals_wide.parquet'))
coverage = pd.read_parquet(download('metadata/coverage_summary.parquet'))
fetch_metadata = json.loads(Path(download('metadata/fetch_metadata.json')).read_text())
snapshot = json.loads(Path(download('metadata/snapshot_manifest.json')).read_text())

tables = {
    'statements_tidy': statements_tidy,
    'statements_wide': statements_wide,
    'earnings': earnings,
    'model_ready': model_ready,
    'coverage': coverage,
}

display(pd.DataFrame([
    {'table': name, 'rows': len(df), 'columns': len(df.columns)}
    for name, df in tables.items()
]))
print('Snapshot information cutoff:', snapshot['information_cutoff'])
print('Statement fetch timestamp:', snapshot['statement_fetch_timestamp_utc'])
print('Latest public report included:', snapshot['latest_public_report_date'])

data/statements_quarterly_tidy.parquet:   0%|          | 0.00/36.1k [00:00<?, ?B/s]

C:\Users\kiera\miniconda3\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\kiera\.cache\huggingface\hub\datasets--cookekieran--mag7-fundamentals. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


data/statements_quarterly_wide.parquet:   0%|          | 0.00/73.1k [00:00<?, ?B/s]

data/earnings_dates_quarterly.parquet:   0%|          | 0.00/9.94k [00:00<?, ?B/s]

data/model_ready_fundamentals_wide.parqu(…):   0%|          | 0.00/75.2k [00:00<?, ?B/s]

metadata/coverage_summary.parquet:   0%|          | 0.00/5.01k [00:00<?, ?B/s]

fetch_metadata.json:   0%|          | 0.00/445 [00:00<?, ?B/s]

snapshot_manifest.json: 0.00B [00:00, ?B/s]

,table,rows,columns
0,statements_tidy,11040,7
1,statements_wide,184,65
2,earnings,120,9
3,model_ready,92,51
4,coverage,14,7


Snapshot information cutoff: 2026-06-01
Statement fetch timestamp: 2026-06-01T18:21:24.561192+00:00
Latest public report included: 2026-05-20


## Understand the table schemas

In [3]:
for name, frame in tables.items():
    print(f'\n{name}:')
    print(frame.columns.tolist())

print('\nCanonical tidy statement sample')
display(statements_tidy.head(12))

print('Earnings/report-date sample')
display(earnings.head(12))


statements_tidy:
['ticker', 'statement_type', 'fiscal_period_end', 'reportedCurrency', 'fetched_at_utc', 'line_item', 'value']

statements_wide:
['ticker', 'statement_type', 'reportedCurrency', 'grossProfit', 'totalRevenue', 'costOfRevenue', 'costofGoodsAndServicesSold', 'operatingIncome', 'sellingGeneralAndAdministrative', 'researchAndDevelopment', 'operatingExpenses', 'investmentIncomeNet', 'netInterestIncome', 'interestIncome', 'interestExpense', 'nonInterestIncome', 'otherNonOperatingIncome', 'depreciation', 'depreciationAndAmortization', 'incomeBeforeTax', 'incomeTaxExpense', 'interestAndDebtExpense', 'netIncomeFromContinuingOperations', 'comprehensiveIncomeNetOfTax', 'ebit', 'ebitda', 'netIncome', 'fiscal_period_end', 'fetched_at_utc', 'totalAssets', 'totalCurrentAssets', 'cashAndCashEquivalentsAtCarryingValue', 'cashAndShortTermInvestments', 'inventory', 'currentNetReceivables', 'totalNonCurrentAssets', 'propertyPlantEquipment', 'accumulatedDepreciationAmortizationPPE', 'intang

,ticker,statement_type,fiscal_period_end,reportedCurrency,fetched_at_utc,line_item,value
0,AAPL,balance_sheet,2023-03-31,USD,2026-06-01T18:21:24.561192+00:00,accumulatedDepreciationAmortizationPPE,6.966800e+10
1,AAPL,balance_sheet,2023-03-31,USD,2026-06-01T18:21:24.561192+00:00,capitalLeaseObligations,NaN
2,AAPL,balance_sheet,2023-03-31,USD,2026-06-01T18:21:24.561192+00:00,cashAndCashEquivalentsAtCarryingValue,2.468700e+10
3,AAPL,balance_sheet,2023-03-31,USD,2026-06-01T18:21:24.561192+00:00,cashAndShortTermInvestments,2.468700e+10
4,AAPL,balance_sheet,2023-03-31,USD,2026-06-01T18:21:24.561192+00:00,commonStock,6.956800e+10
5,AAPL,balance_sheet,2023-03-31,USD,2026-06-01T18:21:24.561192+00:00,commonStockSharesOutstanding,1.584705e+10
6,AAPL,balance_sheet,2023-03-31,USD,2026-06-01T18:21:24.561192+00:00,comprehensiveIncomeNetOfTax,NaN
7,AAPL,balance_sheet,2023-03-31,USD,2026-06-01T18:21:24.561192+00:00,costOfRevenue,NaN
8,AAPL,balance_sheet,2023-03-31,USD,2026-06-01T18:21:24.561192+00:00,costofGoodsAndServicesSold,NaN
9,AAPL,balance_sheet,2023-03-31,USD,2026-06-01T18:21:24.561192+00:00,currentAccountsPayable,4.294500e+10


Earnings/report-date sample


,reportedEPS,estimatedEPS,surprise,surprisePercentage,reportTime,ticker,fiscal_period_end,reported_date,fetched_at_utc
0,1.52,1.43,0.09,6.2937,NaN,AAPL,2022-03-31,2022-04-28,2026-06-01T18:33:13.147809+00:00
1,1.20,1.15,0.05,4.3478,NaN,AAPL,2022-06-30,2022-07-28,2026-06-01T18:33:13.147809+00:00
2,1.29,1.27,0.02,1.5748,NaN,AAPL,2022-09-30,2022-10-27,2026-06-01T18:33:13.147809+00:00
3,1.88,1.95,-0.07,-3.5897,NaN,AAPL,2022-12-31,2023-02-02,2026-06-01T18:33:13.147809+00:00
4,1.52,1.43,0.09,6.2937,NaN,AAPL,2023-03-31,2023-05-04,2026-06-01T18:33:13.147809+00:00
5,1.26,1.19,0.07,5.8824,NaN,AAPL,2023-06-30,2023-08-03,2026-06-01T18:33:13.147809+00:00
6,1.46,1.39,0.07,5.0360,NaN,AAPL,2023-09-30,2023-11-02,2026-06-01T18:33:13.147809+00:00
7,2.18,2.11,0.07,3.3175,NaN,AAPL,2023-12-31,2024-02-01,2026-06-01T18:33:13.147809+00:00
8,1.53,1.50,0.03,2.0000,NaN,AAPL,2024-03-31,2024-05-02,2026-06-01T18:33:13.147809+00:00
9,1.40,1.34,0.06,4.4776,NaN,AAPL,2024-06-30,2024-08-01,2026-06-01T18:33:13.147809+00:00


## Fiscal-period coverage

This describes the accounting periods contained in the dataset. It does **not** describe when those values were knowable.

In [4]:
for frame in (statements_tidy, statements_wide, earnings, model_ready, coverage):
    for column in ['fiscal_period_end', 'reported_date', 'earliest_fiscal_period_end', 'latest_fiscal_period_end']:
        if column in frame.columns:
            frame[column] = pd.to_datetime(frame[column])

display(coverage.sort_values(['ticker', 'statement_type']))

fiscal_coverage = statements_wide.groupby(['ticker', 'statement_type']).agg(
    quarters=('fiscal_period_end', 'nunique'),
    earliest_fiscal_period=('fiscal_period_end', 'min'),
    latest_fiscal_period=('fiscal_period_end', 'max'),
).reset_index()
display(fiscal_coverage)

,ticker,statement_type,quarter_count,earliest_fiscal_period_end,latest_fiscal_period_end,field_count,covers_requested_start_date
0,AAPL,balance_sheet,13,2023-03-31,2026-03-31,36,True
1,AAPL,income_statement,13,2023-03-31,2026-03-31,24,True
2,AMZN,balance_sheet,13,2023-03-31,2026-03-31,36,True
3,AMZN,income_statement,13,2023-03-31,2026-03-31,24,True
4,GOOGL,balance_sheet,13,2023-03-31,2026-03-31,36,True
5,GOOGL,income_statement,13,2023-03-31,2026-03-31,24,True
6,META,balance_sheet,13,2023-03-31,2026-03-31,36,True
7,META,income_statement,13,2023-03-31,2026-03-31,24,True
8,MSFT,balance_sheet,13,2023-03-31,2026-03-31,36,True
9,MSFT,income_statement,13,2023-03-31,2026-03-31,24,True


,ticker,statement_type,quarters,earliest_fiscal_period,latest_fiscal_period
0,AAPL,balance_sheet,13,2023-03-31,2026-03-31
1,AAPL,income_statement,13,2023-03-31,2026-03-31
2,AMZN,balance_sheet,13,2023-03-31,2026-03-31
3,AMZN,income_statement,13,2023-03-31,2026-03-31
4,GOOGL,balance_sheet,13,2023-03-31,2026-03-31
5,GOOGL,income_statement,13,2023-03-31,2026-03-31
6,META,balance_sheet,13,2023-03-31,2026-03-31
7,META,income_statement,13,2023-03-31,2026-03-31
8,MSFT,balance_sheet,13,2023-03-31,2026-03-31
9,MSFT,income_statement,13,2023-03-31,2026-03-31


## Public availability coverage

Alpha Vantage earnings data supplies `reported_date`. This is the date on which the market learned the quarter's result. It is the correct starting point for leakage-safe availability.

Because the dataset does not contain an exact announcement timestamp, this notebook uses a conservative rule:

```text
available_from_date = reported_date + 1 calendar day
```

For a trading model, the strongest implementation would map this to the next valid market trading session.

In [5]:
earnings['reported_date'] = pd.to_datetime(earnings['reported_date']).dt.normalize()
earnings['fiscal_period_end'] = pd.to_datetime(earnings['fiscal_period_end']).dt.normalize()
earnings['available_from_date'] = earnings['reported_date'] + pd.Timedelta(days=1)
earnings['reporting_lag_days'] = (earnings['reported_date'] - earnings['fiscal_period_end']).dt.days

availability_coverage = earnings.groupby('ticker').agg(
    earnings_records=('ticker', 'size'),
    earliest_fiscal_period=('fiscal_period_end', 'min'),
    latest_fiscal_period=('fiscal_period_end', 'max'),
    earliest_reported_date=('reported_date', 'min'),
    latest_reported_date=('reported_date', 'max'),
    earliest_available_from=('available_from_date', 'min'),
    latest_available_from=('available_from_date', 'max'),
    median_reporting_lag_days=('reporting_lag_days', 'median'),
    max_reporting_lag_days=('reporting_lag_days', 'max'),
).reset_index()
display(availability_coverage)

print('Latest publicly available quarter per company')
latest_reports = earnings.sort_values('reported_date').groupby('ticker').tail(1)
display(latest_reports[[
    'ticker', 'fiscal_period_end', 'reported_date', 'available_from_date',
    'reportedEPS', 'estimatedEPS', 'surprisePercentage'
]].sort_values('ticker'))

,ticker,earnings_records,earliest_fiscal_period,latest_fiscal_period,earliest_reported_date,latest_reported_date,earliest_available_from,latest_available_from,median_reporting_lag_days,max_reporting_lag_days
0,AAPL,17,2022-03-31,2026-03-31,2022-04-28,2026-04-30,2022-04-29,2026-05-01,31.0,34
1,AMZN,17,2022-03-31,2026-03-31,2022-04-28,2026-04-29,2022-04-29,2026-04-30,31.0,37
2,GOOGL,17,2022-03-31,2026-03-31,2022-04-26,2026-04-29,2022-04-27,2026-04-30,26.0,35
3,META,17,2022-03-31,2026-03-31,2022-04-27,2026-04-29,2022-04-28,2026-04-30,29.0,32
4,MSFT,17,2022-03-31,2026-03-31,2022-04-26,2026-04-29,2022-04-27,2026-04-30,28.0,30
5,NVDA,18,2022-01-31,2026-04-30,2022-02-16,2026-05-20,2022-02-17,2026-05-21,22.5,28
6,TSLA,17,2022-03-31,2026-03-31,2022-04-20,2026-04-22,2022-04-21,2026-04-23,22.0,29


Latest publicly available quarter per company


,ticker,fiscal_period_end,reported_date,available_from_date,reportedEPS,estimatedEPS,surprisePercentage
16,AAPL,2026-03-31,2026-04-30,2026-05-01,2.01,1.94,3.6082
33,AMZN,2026-03-31,2026-04-29,2026-04-30,2.78,1.65,68.4848
50,GOOGL,2026-03-31,2026-04-29,2026-04-30,5.11,2.53,101.9763
67,META,2026-03-31,2026-04-29,2026-04-30,7.31,6.82,7.1848
84,MSFT,2026-03-31,2026-04-29,2026-04-30,4.27,4.09,4.4010
102,NVDA,2026-04-30,2026-05-20,2026-05-21,1.87,1.77,5.6497
119,TSLA,2026-03-31,2026-04-22,2026-04-23,0.41,0.35,17.1429


## Match statements to release dates

Each income statement and balance sheet should match one earnings-release record using `(ticker, fiscal_period_end)`. Missing matches would require a conservative fallback date and should be clearly flagged.

In [6]:
release_dates = earnings[[
    'ticker', 'fiscal_period_end', 'reported_date', 'available_from_date', 'reporting_lag_days'
]].drop_duplicates(['ticker', 'fiscal_period_end'])

statement_availability = statements_wide.merge(
    release_dates,
    on=['ticker', 'fiscal_period_end'],
    how='left',
    validate='many_to_one',
)
statement_availability['has_release_date'] = statement_availability['reported_date'].notna()

match_audit = statement_availability.groupby(['ticker', 'statement_type']).agg(
    statement_rows=('ticker', 'size'),
    matched_release_dates=('has_release_date', 'sum'),
    missing_release_dates=('has_release_date', lambda values: (~values).sum()),
    first_safe_availability=('available_from_date', 'min'),
    last_safe_availability=('available_from_date', 'max'),
).reset_index()
display(match_audit)

missing = statement_availability[~statement_availability['has_release_date']]
print('Statement rows missing a release-date match:', len(missing))
if len(missing):
    display(missing[['ticker', 'statement_type', 'fiscal_period_end']])

,ticker,statement_type,statement_rows,matched_release_dates,missing_release_dates,first_safe_availability,last_safe_availability
0,AAPL,balance_sheet,13,13,0,2023-05-05,2026-05-01
1,AAPL,income_statement,13,13,0,2023-05-05,2026-05-01
2,AMZN,balance_sheet,13,13,0,2023-04-28,2026-04-30
3,AMZN,income_statement,13,13,0,2023-04-28,2026-04-30
4,GOOGL,balance_sheet,13,13,0,2023-04-26,2026-04-30
5,GOOGL,income_statement,13,13,0,2023-04-26,2026-04-30
6,META,balance_sheet,13,13,0,2023-04-27,2026-04-30
7,META,income_statement,13,13,0,2023-04-27,2026-04-30
8,MSFT,balance_sheet,13,13,0,2023-04-26,2026-04-30
9,MSFT,income_statement,13,13,0,2023-04-26,2026-04-30


Statement rows missing a release-date match: 0


## Demonstrate the leakage risk

The table below shows how many days separate the quarter end from the first conservative availability date. During this interval, using the new quarter's values would leak information.

In [7]:
leakage_windows = release_dates.copy()
leakage_windows['unsafe_window_days'] = (
    leakage_windows['available_from_date'] - leakage_windows['fiscal_period_end']
).dt.days

display(leakage_windows.groupby('ticker').agg(
    observations=('ticker', 'size'),
    median_unsafe_window_days=('unsafe_window_days', 'median'),
    shortest_unsafe_window_days=('unsafe_window_days', 'min'),
    longest_unsafe_window_days=('unsafe_window_days', 'max'),
).reset_index())

print('Example release timeline')
display(leakage_windows.sort_values('reported_date').tail(14)[[
    'ticker', 'fiscal_period_end', 'reported_date', 'available_from_date',
    'reporting_lag_days', 'unsafe_window_days'
]])

,ticker,observations,median_unsafe_window_days,shortest_unsafe_window_days,longest_unsafe_window_days
0,AAPL,17,32.0,28,35
1,AMZN,17,32.0,27,38
2,GOOGL,17,27.0,24,36
3,META,17,30.0,25,33
4,MSFT,17,29.0,25,31
5,NVDA,18,23.5,17,29
6,TSLA,17,23.0,19,30


Example release timeline


,ticker,fiscal_period_end,reported_date,available_from_date,reporting_lag_days,unsafe_window_days
118,TSLA,2025-12-31,2026-01-28,2026-01-29,28,29
83,MSFT,2025-12-31,2026-01-28,2026-01-29,28,29
66,META,2025-12-31,2026-01-28,2026-01-29,28,29
15,AAPL,2025-12-31,2026-01-29,2026-01-30,29,30
49,GOOGL,2025-12-31,2026-02-04,2026-02-05,35,36
32,AMZN,2025-12-31,2026-02-05,2026-02-06,36,37
101,NVDA,2026-01-31,2026-02-25,2026-02-26,25,26
119,TSLA,2026-03-31,2026-04-22,2026-04-23,22,23
50,GOOGL,2026-03-31,2026-04-29,2026-04-30,29,30
33,AMZN,2026-03-31,2026-04-29,2026-04-30,29,30


## Leakage-safe as-of join demonstration

For every prediction date, use only the most recent statement whose `available_from_date` is no later than that prediction date. `merge_asof(..., direction='backward')` is a suitable implementation.

This demonstration creates daily dates through the June 1 cutoff and verifies that no future statement is attached.

In [8]:
quarterly_release_table = release_dates.sort_values(['available_from_date', 'ticker']).copy()
calendar = pd.MultiIndex.from_product(
    [
        pd.date_range('2023-01-01', snapshot['information_cutoff'], freq='D'),
        sorted(quarterly_release_table['ticker'].unique()),
    ],
    names=['prediction_date', 'ticker'],
).to_frame(index=False).sort_values(['prediction_date', 'ticker'])

daily_asof = pd.merge_asof(
    calendar,
    quarterly_release_table,
    by='ticker',
    left_on='prediction_date',
    right_on='available_from_date',
    direction='backward',
    allow_exact_matches=True,
)

leaks = daily_asof[
    daily_asof['available_from_date'].notna()
    & (daily_asof['available_from_date'] > daily_asof['prediction_date'])
]
print('Rows tested:', f'{len(daily_asof):,}')
print('Rows using information before it became available:', len(leaks))
assert leaks.empty

print('\nStatements available on June 1, 2026')
display(daily_asof[daily_asof['prediction_date'].eq(pd.Timestamp('2026-06-01'))][[
    'ticker', 'prediction_date', 'fiscal_period_end', 'reported_date', 'available_from_date'
]])

Rows tested: 8,736
Rows using information before it became available: 0

Statements available on June 1, 2026


,ticker,prediction_date,fiscal_period_end,reported_date,available_from_date
8729,AAPL,2026-06-01,2026-03-31,2026-04-30,2026-05-01
8730,AMZN,2026-06-01,2026-03-31,2026-04-29,2026-04-30
8731,GOOGL,2026-06-01,2026-03-31,2026-04-29,2026-04-30
8732,META,2026-06-01,2026-03-31,2026-04-29,2026-04-30
8733,MSFT,2026-06-01,2026-03-31,2026-04-29,2026-04-30
8734,NVDA,2026-06-01,2026-04-30,2026-05-20,2026-05-21
8735,TSLA,2026-06-01,2026-03-31,2026-04-22,2026-04-23


## Inspect line items and missingness

In [9]:
line_item_summary = statements_tidy.groupby(['statement_type', 'line_item']).agg(
    observations=('value', 'size'),
    non_null_values=('value', 'count'),
    companies=('ticker', 'nunique'),
    earliest_fiscal_period=('fiscal_period_end', 'min'),
    latest_fiscal_period=('fiscal_period_end', 'max'),
).reset_index()
line_item_summary['missing_share'] = 1 - (
    line_item_summary['non_null_values'] / line_item_summary['observations']
)

print('Most complete line items')
display(line_item_summary.sort_values(['missing_share', 'statement_type', 'line_item']).head(25))

print('Least complete line items')
display(line_item_summary.sort_values('missing_share', ascending=False).head(25))

Most complete line items


,statement_type,line_item,observations,non_null_values,companies,earliest_fiscal_period,latest_fiscal_period,missing_share
2,balance_sheet,cashAndCashEquivalentsAtCarryingValue,92,92,7,2023-01-31,2026-04-30,0.0
3,balance_sheet,cashAndShortTermInvestments,92,92,7,2023-01-31,2026-04-30,0.0
5,balance_sheet,commonStockSharesOutstanding,92,92,7,2023-01-31,2026-04-30,0.0
9,balance_sheet,currentAccountsPayable,92,92,7,2023-01-31,2026-04-30,0.0
12,balance_sheet,currentNetReceivables,92,92,7,2023-01-31,2026-04-30,0.0
30,balance_sheet,longTermDebt,92,92,7,2023-01-31,2026-04-30,0.0
40,balance_sheet,otherCurrentLiabilities,92,92,7,2023-01-31,2026-04-30,0.0
42,balance_sheet,otherNonCurrentLiabilities,92,92,7,2023-01-31,2026-04-30,0.0
46,balance_sheet,retainedEarnings,92,92,7,2023-01-31,2026-04-30,0.0
48,balance_sheet,shortLongTermDebtTotal,92,92,7,2023-01-31,2026-04-30,0.0


Least complete line items


,statement_type,line_item,observations,non_null_values,companies,earliest_fiscal_period,latest_fiscal_period,missing_share
6,balance_sheet,comprehensiveIncomeNetOfTax,92,0,7,2023-01-31,2026-04-30,1.0
7,balance_sheet,costOfRevenue,92,0,7,2023-01-31,2026-04-30,1.0
19,balance_sheet,grossProfit,92,0,7,2023-01-31,2026-04-30,1.0
15,balance_sheet,depreciationAndAmortization,92,0,7,2023-01-31,2026-04-30,1.0
13,balance_sheet,deferredRevenue,92,0,7,2023-01-31,2026-04-30,1.0
14,balance_sheet,depreciation,92,0,7,2023-01-31,2026-04-30,1.0
10,balance_sheet,currentDebt,92,0,7,2023-01-31,2026-04-30,1.0
8,balance_sheet,costofGoodsAndServicesSold,92,0,7,2023-01-31,2026-04-30,1.0
21,balance_sheet,incomeTaxExpense,92,0,7,2023-01-31,2026-04-30,1.0
20,balance_sheet,incomeBeforeTax,92,0,7,2023-01-31,2026-04-30,1.0


## Conclusions for modelling

- The dataset snapshot contains Alpha Vantage fundamentals available by June 1, 2026.
- Most companies' latest available statement describes the quarter ending March 31, 2026; NVIDIA's latest describes April 30, 2026.
- Statement values must become usable based on `reported_date`, not `fiscal_period_end`.
- With no announcement timestamps, using `reported_date + 1 day` is a conservative rule. Mapping to the next market trading session is better.
- Use a backward as-of join so each prediction receives only the latest already-public statement.
- Keep the repository revision hash shown at the top with experiment metadata for reproducibility.